# BRIDGE-Net — six baseline segmentation models
Standalone benchmark notebook. Uses the existing preprocessed dataset, fixed patient partitions, independent foreground channels and known-label vectors. Run top to bottom. Default protocol: random initialization for every architecture, identical optimizer budget, validation-only model selection, held-out test evaluation explicitly gated. No BRIDGE-Net training here. UCTransNet is the published model name (previously called UCTTransNet in our discussion).

## 1. Dependencies
Use a GPU environment with a compatible PyTorch/torchvision installation. Install packages once; restart kernel if needed. Official repositories are downloaded only when you enable the setup cell.

In [1]:
%pip install segmentation-models-pytorch==0.5.0 timm==1.0.15 einops ml-collections scipy pandas matplotlib pillow opencv-python tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 11.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [segmentation-models-pytorch]e-hub]

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json, random, time, hashlib, copy, gc, math, subprocess, sys, importlib, types
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DATA = Path('bridge_preprocessed_baseline').resolve()
OUT = Path('bridge_baseline_benchmarks').resolve()
REPOS = Path('baseline_repositories').resolve()
MODELS = ['UNet','DeepLabV3Plus','SwinUNet','SegFormer','TransUNet','UCTransNet']
SEEDS = [42] # final study: e.g. [42, 123, 2026]
EPOCHS = 200
BATCH = 2 # same for every model; change globally if GPU memory is insufficient
LR = 1e-4
WD = 1e-4
WARMUP = 5
PATIENCE = 25
AMP = True
AUGMENT = True
WEIGHTED_SAMPLING = True
CLASS_WEIGHTS = True
THRESHOLD = 0.5
DOWNLOAD_REPOS = False
RUN_SMOKE = False
RUN_TRAINING = False
RUN_TEST = False # enable only after all architecture and hyperparameter choices are frozen
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DATA != OUT and DATA not in OUT.parents
OUT.mkdir(parents=True,exist_ok=True)
print(DEVICE, DATA)


cpu /Users/bash/Documents/Phd code/bash_phd/labelbox_export_segmentation/bridge_preprocessed_baseline


/Users/bash/Documents/Phd code/bash_phd/labelbox_export_segmentation/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Dataset checks and frozen partitions

In [3]:
def read_json(p): return json.loads(Path(p).read_text())
for f in ['class_mapping.json','split_manifest.csv','normalisation.json','class_weights.json','sampling_weights.csv','hashes.csv']:
    if not (DATA/f).is_file(): raise FileNotFoundError(DATA/f)
classes = read_json(DATA/'class_mapping.json'); K=len(classes)
assert K>1 and sorted(classes.values())==list(range(K))
names=sorted(classes,key=classes.get)
manifest=pd.read_csv(DATA/'split_manifest.csv',dtype=str)
assert not manifest.sample_id.duplicated().any()
assert manifest.patient_id.notna().all() and manifest.patient_id.str.strip().ne('').all()
assert set(manifest.split)=={'train','val','test'}
assert manifest.groupby('patient_id').split.nunique().max()==1
hashes=pd.read_csv(DATA/'hashes.csv',dtype=str)
h=manifest.merge(hashes[['sample_id','sha256_pixels']],on='sample_id',how='left',validate='one_to_one')
assert h.sha256_pixels.notna().all() and h.groupby('sha256_pixels').split.nunique().max()==1
ids={p:manifest.loc[manifest.split==p,'sample_id'].tolist() for p in ['train','val','test']}
for p, samples in ids.items():
    assert set(samples)==set(read_json(DATA/'splits'/f'{p}.json'))
    for sid in samples:
        for f in [DATA/'images'/f'{sid}.png',DATA/'multilabel_masks'/f'{sid}.npy',DATA/'known'/f'{sid}.npy']:
            if not f.is_file(): raise FileNotFoundError(f)
normal=read_json(DATA/'normalisation.json'); MU=float(normal['mean']); STD=float(normal['std']); C=3 if normal['rgb'] else 1
assert STD>0
with Image.open(DATA/'images'/f"{ids['train'][0]}.png") as im: W,H=im.size
assert H==W and H%32==0; SIZE=H
print('Classes:',K,'Input:',C,SIZE, 'Partitions:',{p:len(v) for p,v in ids.items()})


Classes: 23 Input: 1 512 Partitions: {'train': 2810, 'val': 591, 'test': 597}


## 3. Shared data loader and synchronized augmentation
Full-image training for all six models; patch sampling is deliberately disabled to preserve global context and fixed-size transformer compatibility. Apply the same protocol to BRIDGE-Net for the architecture-only comparison. No test augmentation.

In [4]:
import cv2
weights=read_json(DATA/'class_weights.json')
cw=np.asarray(weights['class_weights'],np.float32); pw=np.asarray(weights['bce_pos_weights'],np.float32)
assert cw.shape==pw.shape==(K,) and np.isfinite(cw).all() and np.isfinite(pw).all()
if not CLASS_WEIGHTS: cw[:]=1; pw[:]=1
sw=pd.read_csv(DATA/'sampling_weights.csv').set_index('sample_id').weight
assert sw.index.is_unique and set(ids['train'])<=set(sw.index)
assert np.isfinite(sw.loc[ids['train']]).all() and (sw.loc[ids['train']]>0).all()
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
class CTDataset(Dataset):
    def __init__(self,split): self.ids=ids[split]; self.split=split
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        sid=self.ids[i]
        with Image.open(DATA/'images'/f'{sid}.png') as im: a=np.array(im.convert('L'))
        y=np.load(DATA/'multilabel_masks'/f'{sid}.npy',allow_pickle=False); k=np.load(DATA/'known'/f'{sid}.npy',allow_pickle=False)
        assert a.shape==(SIZE,SIZE) and y.shape==(K,SIZE,SIZE) and k.shape==(K,),sid
        assert np.isin(y,[0,1]).all() and np.isin(k,[0,1]).all(),sid
        assert not ((y.any(axis=(1,2))) & (k==0)).any(),sid
        if self.split=='train' and AUGMENT:
            orig_a,orig_y=a,y
            mat=cv2.getRotationMatrix2D((SIZE/2,SIZE/2),random.uniform(-10,10),random.uniform(.9,1.1)); mat[:,2]+=np.random.uniform(-.05,.05,2)*SIZE
            a=cv2.warpAffine(a,mat,(SIZE,SIZE),flags=cv2.INTER_LINEAR)
            y=np.stack([cv2.warpAffine(z.astype(np.uint8),mat,(SIZE,SIZE),flags=cv2.INTER_NEAREST) for z in y])
            # Reject geometry that deletes an entire positive channel.
            if (orig_y.any((1,2)) & ~y.any((1,2))).any(): a,y=orig_a,orig_y
            a=np.clip(a.astype(np.float32)*random.uniform(.9,1.1)+random.uniform(-10,10),0,255)
        x=(a.astype(np.float32)/255-MU)/STD
        return torch.from_numpy(np.repeat(x[None],C,axis=0)),torch.from_numpy(y.astype(np.float32)),torch.from_numpy(k.astype(np.float32)),sid

def make_loader(split,seed):
    ds=CTDataset(split); g=torch.Generator().manual_seed(seed)
    sampler=WeightedRandomSampler(sw.loc[ds.ids].to_numpy(),len(ds),replacement=True,generator=g) if split=='train' and WEIGHTED_SAMPLING else None
    # Drop a singleton training batch to avoid BatchNorm failure in pooled heads.
    return DataLoader(ds,batch_size=BATCH,shuffle=split=='train' and sampler is None,sampler=sampler,generator=g,num_workers=0,pin_memory=DEVICE.type=='cuda',drop_last=split=='train' and len(ds)%BATCH==1)
assert BATCH>=2 and len(ids['train'])>=BATCH


## 4. Model sources and setup
U-Net and DeepLabV3+ use segmentation-models-pytorch with ResNet34; SegFormer uses the Hugging Face MiT-B0 configuration. The other three use author repositories. Clone and inspect the code before executing it. Record commits; use the same commits for all repetitions. Random initialization avoids unequal pretrained checkpoints. Official architectures are adapted for 512×512 and multilabel logits.

Sources: https://github.com/qubvel-org/segmentation_models.pytorch ; https://github.com/HuCaoFighting/Swin-Unet ; https://github.com/Beckschen/TransUNet ; https://github.com/McGregorWwww/UCTransNet ; https://huggingface.co/docs/transformers/model_doc/segformer

In [16]:
URLS={'SwinUNet':'https://github.com/HuCaoFighting/Swin-Unet.git','TransUNet':'https://github.com/Beckschen/TransUNet.git','UCTransNet':'https://github.com/McGregorWwww/UCTransNet.git'}
if DOWNLOAD_REPOS:
    REPOS.mkdir(parents=True,exist_ok=True)
    for name,url in URLS.items():
        if not (REPOS/name).exists(): subprocess.run(['git','clone',url,str(REPOS/name)],check=True)
for name in URLS:
    if (REPOS/name/'.git').exists(): print(name,subprocess.check_output(['git','-C',str(REPOS/name),'rev-parse','HEAD'],text=True).strip())
%pip install transformers==4.49.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 7.1 MB/s  0:00:01m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 9.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 1.8 MB/s  0:00:01m eta 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 2.4.0
    Uninstalling huggingface_hub-2.4.0:
      Successfully uninstalled huggingface_hub-2.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]ub]

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
def repo_module(model,relative):
    # Isolate similarly named packages (e.g. networks) across repositories.
    root=REPOS/model
    if not root.is_dir(): raise FileNotFoundError(f'Clone {URLS[model]} into {root} first')
    alias='_bridge_'+model.lower()
    if alias not in sys.modules:
        pkg=types.ModuleType(alias); pkg.__path__=[str(root)]; sys.modules[alias]=pkg
    return importlib.import_module(alias+'.'+relative)
class LogitAdapter(nn.Module):
    def __init__(self,net): super().__init__(); self.net=net
    def forward(self,x):
        y=self.net(x)
        if hasattr(y,'logits'): y=y.logits
        if not torch.is_tensor(y) or y.ndim!=4 or y.shape[1]!=K: raise RuntimeError('Unexpected model output; inspect adapter')
        if y.shape[-2:]!=x.shape[-2:]: y=F.interpolate(y,size=x.shape[-2:],mode='bilinear',align_corners=False)
        return y


### 4a. CNN baselines

In [18]:
def build_unet():
    import segmentation_models_pytorch as smp
    return smp.Unet(encoder_name='resnet34',encoder_weights=None,in_channels=C,classes=K,activation=None)
def build_deeplab():
    import segmentation_models_pytorch as smp
    return smp.DeepLabV3Plus(encoder_name='resnet34',encoder_weights=None,in_channels=C,classes=K,activation=None)


### 4b. Transformer baselines

In [19]:
def build_swin():
    mod=repo_module('SwinUNet','networks.swin_transformer_unet_skip_expand_decoder_sys')
    # Window 8 divides the 512-input feature grids; adjust for other resolutions.
    window=8 if SIZE%256==0 else (7 if SIZE%224==0 else 4 if SIZE%128==0 else 2)
    return mod.SwinTransformerSys(img_size=SIZE,in_chans=C,num_classes=K,embed_dim=96,depths=[2,2,6,2],num_heads=[3,6,12,24],window_size=window)
def build_segformer():
    from transformers import SegformerConfig, SegformerForSemanticSegmentation
    cfg=SegformerConfig(num_channels=C,num_labels=K,depths=[2,2,2,2],hidden_sizes=[32,64,160,256],decoder_hidden_size=256)
    return SegformerForSemanticSegmentation(cfg)


### 4c. Hybrid baselines

In [20]:
def build_transunet():
    mod=repo_module('TransUNet','networks.vit_seg_modeling')
    cfg=copy.deepcopy(mod.CONFIGS['R50-ViT-B_16']); cfg.n_classes=K; cfg.n_skip=3; cfg.patches.grid=(SIZE//16,SIZE//16)
    # Official hybrid backbone internally repeats grayscale to RGB.
    return mod.VisionTransformer(cfg,img_size=SIZE,num_classes=K)
def build_uctransnet():
    from ml_collections import ConfigDict
    mod=repo_module('UCTransNet','nets.UCTransNet')
    # Reproduce the author configuration without importing Config.py (GPU side effects).
    cfg=ConfigDict(dict(KV_size=960,expand_ratio=4,patch_sizes=[16,8,4,2],base_channel=64,n_classes=K,transformer=dict(num_heads=4,num_layers=4,embeddings_dropout_rate=.1,attention_dropout_rate=.1,dropout_rate=0.)))
    return mod.UCTransNet(cfg,n_channels=C,n_classes=K,img_size=SIZE,vis=False)
BUILDERS={'UNet':build_unet,'DeepLabV3Plus':build_deeplab,'SwinUNet':build_swin,'SegFormer':build_segformer,'TransUNet':build_transunet,'UCTransNet':build_uctransnet}
def build(name): return LogitAdapter(BUILDERS[name]())


## 5. Common masked Dice + BCE and metrics
Metrics aggregate only known labels. Macro Dice averages classes with positive ground truth in the evaluated partition, and separately reports false-positive rates on known-negative images. Unsupported classes are NaN, not perfect scores. All models use the same fixed threshold.

In [21]:
def criterion(logits,y,k):
    logits=logits.float(); y=y.float(); valid=k.float(); v=valid[:,:,None,None]
    pweight=torch.as_tensor(pw,device=y.device)[None,:,None,None]
    bce=(F.binary_cross_entropy_with_logits(logits,y,pos_weight=pweight,reduction='none')*v).sum()/(v.sum()*y.shape[-1]*y.shape[-2]).clamp_min(1)
    p=logits.sigmoid(); d=1-(2*(p*y).sum((2,3))+1e-6)/((p+y).sum((2,3))+1e-6)
    w=valid*torch.as_tensor(cw,device=y.device)[None,:]
    return bce+(d*w).sum()/w.sum().clamp_min(1)
class Metrics:
    def __init__(self): self.counts=np.zeros((6,K),np.float64)
    def update(self,z,y,k):
        p=z.sigmoid()>=THRESHOLD; t=y.bool(); v=k.bool(); m=v[:,:,None,None]
        neg=v & ~t.any((2,3)); dims=(0,2,3)
        vals=[(p&t&m).sum(dims),(p&~t&m).sum(dims),(~p&t&m).sum(dims),v.sum(0),neg.sum(0),(neg&p.any((2,3))).sum(0)]
        self.counts+=torch.stack(vals).cpu().numpy()
    def report(self):
        tp,fp,fn,known,neg,negfp=self.counts
        def div(a,b): return np.divide(a,b,out=np.full(K,np.nan),where=b>0)
        positive=tp+fn>0
        dice=div(2*tp,2*tp+fp+fn); dice[~positive]=np.nan
        iou=div(tp,tp+fp+fn); iou[~positive]=np.nan
        return pd.DataFrame(dict(biomarker=names,dice=dice,iou=iou,sensitivity=div(tp,tp+fn),precision=div(tp,tp+fp),known_images=known,positive_pixels=tp+fn,negative_images=neg,negative_image_fp_rate=div(negfp,neg)))


## 6. Forward/backward smoke tests (one model at a time)
Enable RUN_SMOKE in the configuration. Failures are recorded; never silently substitute another architecture.

In [22]:
if RUN_SMOKE:
    results=[]
    for name in MODELS:
        model=None
        try:
            seed_all(SEEDS[0]); model=build(name).to(DEVICE).train()
            x,y,k,_=next(iter(make_loader('train',SEEDS[0])))
            x,y,k=[t.to(DEVICE) for t in (x,y,k)]
            z=model(x); assert z.shape==y.shape
            loss=criterion(z,y,k); assert torch.isfinite(loss); loss.backward()
            grads=[p.grad for p in model.parameters() if p.grad is not None]
            assert grads and all(torch.isfinite(g).all().item() for g in grads)
            results.append(dict(model=name,status='passed',loss=float(loss),parameters=sum(p.numel() for p in model.parameters())))
        except Exception as e: results.append(dict(model=name,status='failed',error=repr(e)))
        finally:
            del model; gc.collect()
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
    pd.DataFrame(results).to_csv(OUT/'smoke_results.csv',index=False); display(pd.DataFrame(results))


## 7. Shared training and validation loop
Best checkpoint selected by validation macro Dice only. Training logs, latest and best checkpoints are saved. Fresh runs refuse to overwrite existing run directories; restart an interrupted run using a new output directory or archive its existing folder. Latest checkpoint includes optimizer state for manual recovery.

In [23]:
def epoch_pass(model,loader,opt=None,scaler=None):
    training=opt is not None; model.train(training); metric=Metrics(); total=0.; count=0
    for x,y,k,_ in loader:
        x,y,k=[t.to(DEVICE,non_blocking=True) for t in (x,y,k)]
        if not k.any(): continue
        if training: opt.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=DEVICE.type,enabled=AMP and DEVICE.type=='cuda'):
                z=model(x)
            loss=criterion(z,y,k)
            if not torch.isfinite(loss): raise FloatingPointError('Nonfinite loss')
            if training:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(opt); scaler.update()
        with torch.no_grad(): metric.update(z.detach().float(),y,k)
        total+=loss.item()*len(x); count+=len(x)
    if not count: raise RuntimeError('No known supervision in partition')
    tab=metric.report(); return total/count,tab

def save_ckpt(obj,path):
    tmp=path.with_suffix('.tmp'); torch.save(obj,tmp); tmp.replace(path)

def train_model(name,seed):
    seed_all(seed); model=build(name).to(DEVICE)
    folder=OUT/name/f'seed_{seed}'; folder.mkdir(parents=True,exist_ok=False)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    def scale(e): return (e+1)/max(1,WARMUP) if e<WARMUP else .5*(1+math.cos(math.pi*(e-WARMUP)/max(1,EPOCHS-WARMUP)))
    scheduler=torch.optim.lr_scheduler.LambdaLR(opt,scale)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP and DEVICE.type=='cuda')
    tr=make_loader('train',seed); va=make_loader('val',seed)
    best=-float('inf'); stale=0; history=[]; start=time.time()
    for epoch in range(1,EPOCHS+1):
        tl,tm=epoch_pass(model,tr,opt,scaler); vl,vm=epoch_pass(model,va)
        score=float(vm.dice.mean()); assert np.isfinite(score),'No positive known validation class'
        history.append(dict(epoch=epoch,train_loss=tl,val_loss=vl,train_dice=tm.dice.mean(),val_dice=score,lr=opt.param_groups[0]['lr']))
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False); scheduler.step()
        improved=score>best
        if improved: best=score; stale=0; vm.to_csv(folder/'best_val_per_class.csv',index=False)
        else: stale+=1
        state=dict(model=model.state_dict(),optimizer=opt.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),epoch=epoch,best=best,classes=classes)
        save_ckpt(state,folder/'latest.pt')
        if improved: save_ckpt(state,folder/'best.pt')
        print(name,seed,epoch,round(tl,4),round(score,4))
        if stale>=PATIENCE: break
    result=dict(model=name,seed=seed,best_val_dice=best,parameters=sum(p.numel() for p in model.parameters()),seconds=time.time()-start,epochs=epoch)
    (folder/'summary.json').write_text(json.dumps(result,indent=2)); return result


## 8. Run the six models
Set RUN_TRAINING=True. Use a subset of MODELS to run one architecture at a time. Record dataset fingerprint and package versions. All models must finish before claiming a six-model comparison.

In [24]:
import importlib.metadata as im
protocol=dict(dataset=str(DATA),split_sha256=hashlib.sha256((DATA/'split_manifest.csv').read_bytes()).hexdigest(),mapping=classes,size=SIZE,channels=C,seeds=SEEDS,epochs=EPOCHS,batch=BATCH,lr=LR,wd=WD,warmup=WARMUP,patience=PATIENCE,augmentation=AUGMENT,weighted_sampling=WEIGHTED_SAMPLING,class_weights=CLASS_WEIGHTS,amp=AMP,threshold=THRESHOLD,initialization='random')
protocol['packages']={p:im.version(p) for p in ['torch','numpy','pandas','segmentation-models-pytorch','transformers','timm'] if p in {d.metadata['Name'] for d in im.distributions()}}
protocol['commits']={name:subprocess.check_output(['git','-C',str(REPOS/name),'rev-parse','HEAD'],text=True).strip() for name in URLS if (REPOS/name/'.git').exists()}
if RUN_TRAINING:
    path=OUT/'protocol.json'
    if path.exists() and read_json(path)!=protocol: raise RuntimeError('Protocol changed: choose a new output directory')
    path.write_text(json.dumps(protocol,indent=2)); failures=[]
    for name in MODELS:
        for seed in SEEDS:
            folder=OUT/name/f'seed_{seed}'
            if (folder/'summary.json').exists(): continue
            try: train_model(name,seed)
            except Exception as e:
                failures.append(dict(model=name,seed=seed,error=repr(e))); print(failures[-1])
            finally:
                gc.collect()
                if DEVICE.type=='cuda': torch.cuda.empty_cache()
    pd.DataFrame(failures,columns=['model','seed','error']).to_csv(OUT/'training_failures.csv',index=False)


## 9. Validation comparison and learning curves

In [25]:
summaries=[read_json(p) for p in OUT.glob('*/seed_*/summary.json')]
if summaries:
    summary=pd.DataFrame(summaries); summary.to_csv(OUT/'validation_runs.csv',index=False)
    agg=summary.groupby('model').agg(mean_val_dice=('best_val_dice','mean'),std_val_dice=('best_val_dice','std'),runs=('seed','count'),parameters=('parameters','mean')).sort_values('mean_val_dice',ascending=False)
    agg.to_csv(OUT/'validation_comparison.csv'); display(agg)
    fig,ax=plt.subplots(figsize=(10,5)); agg.mean_val_dice.plot.bar(ax=ax); ax.set_ylabel('Validation macro Dice'); ax.set_ylim(0,1); fig.tight_layout(); fig.savefig(OUT/'validation_comparison.png',dpi=300); plt.show()
    for name in summary.model.unique():
        fig,ax=plt.subplots(figsize=(7,4))
        for p in (OUT/name).glob('seed_*/history.csv'):
            h=pd.read_csv(p); ax.plot(h.epoch,h.val_dice,label=p.parent.name)
        ax.set(title=name,xlabel='Epoch',ylabel='Validation macro Dice'); ax.legend(); fig.tight_layout(); fig.savefig(OUT/f'{name}_learning_curve.png',dpi=300); plt.show()
else: print('No completed training runs yet.')


No completed training runs yet.


## 10. Locked test evaluation
Set RUN_TEST=True only once model selection and protocol are finalized. Test metrics are not used to select thresholds or architecture. Evaluate all completed planned runs, not just the strongest seed.

In [26]:
if RUN_TEST:
    assert read_json(OUT/'protocol.json')==protocol,'Protocol mismatch'
    assert all((OUT/m/f'seed_{s}'/'summary.json').exists() for m in MODELS for s in SEEDS),'Complete planned runs first'
    rows=[]
    for name in MODELS:
        for seed in SEEDS:
            folder=OUT/name/f'seed_{seed}'; model=build(name).to(DEVICE)
            ck=torch.load(folder/'best.pt',map_location='cpu',weights_only=True); assert ck['classes']==classes
            model.load_state_dict(ck['model']); loss,tab=epoch_pass(model,make_loader('test',seed))
            tab.to_csv(folder/'test_per_class.csv',index=False)
            rows.append(dict(model=name,seed=seed,test_loss=loss,macro_dice=tab.dice.mean(),macro_iou=tab.iou.mean(),macro_sensitivity=tab.sensitivity.mean(),macro_precision=tab.precision.mean()))
            del model,ck; gc.collect()
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
    results=pd.DataFrame(rows); results.to_csv(OUT/'test_runs.csv',index=False)
    result=results.groupby('model').agg({'macro_dice':['mean','std'],'macro_iou':['mean','std'],'macro_sensitivity':['mean','std'],'macro_precision':['mean','std']}); result.to_csv(OUT/'test_comparison.csv'); display(result)


## 11. Interpretation and reporting
- Report the exact implementations, initialization, resolution, parameter counts, seeds, training budget, and source revisions. U-Net here is a ResNet34 encoder variant; Swin window size is adapted to image dimensions. Do not claim exact paper replication.
- Compare BRIDGE-Net with identical preprocessing, training-time strategies, loss, and split definitions before running architecture-specific improvements separately.
- Pixel-pooled per-class Dice is averaged across positive-supported classes. Report missing classes and known-negative false positives. Patient-level confidence intervals and hospital-specific evaluation can be added for the final manuscript; not implemented here.
- GPU requirements differ substantially; reduce the common batch size or adopt a common smaller resolution if necessary, and rerun all models under the revised protocol. UCTransNet at 512 may be particularly expensive. No automatic downsampling or silent model substitutions.
- External repository compatibility and actual model forward/backward execution must be verified in your environment. Run the smoke tests first. Never change fixed patient partitions after using test results.